## Bayesian backoff + TT adapter (eval launcher)

**What:** **Self-contained** eval A/B on fixed validation slices—same checkpoint, **cache-only candidate** vs **BayesianBackoffCache + TestTimeAdapter**. Cell **1** builds the baseline checkpoint (checkpoint-only train) so you do not need a separate training runbook. **`progress.csv`** here is **slice-level eval** (A/B schema), not the training staging CSV.

**Paths & run:** **`/content/pg_eval_adapter_staged`** (adapter eval **`progress.csv`**) · **`/content/pg/results/baseline_staged`** (checkpoint-only baseline; mirrored to **`MyDrive/baseline_staged`** / **`MyDrive/pg_ttadapter_staged`** when Drive is mounted, same idea as the int6 book) · **GPU** for auto-baseline in cell 1 · Run All · Stage B if **`PROCEED_STAGE_B: YES`** · clean slate: **`export FRESH_PROGRESS_CSV=1`** then cell 1.



---
## Code cell 1: Setup
**Drive** mounts in this **Python** cell (same pattern as **`colab_int6_gps_int4_mlp_train.ipynb`** — not inside a bash subprocess). Then bash clones **`openai-pg`** plus **two** trees under **`/content/pg_candidate`** (int6 branch) and **`/content/pg_ttadapter`** (Bayesian adapter branch), fineweb shard, Colab patches, **`run_one.sh`**, optional **AUTO_BASELINE** checkpoint train, and adapter **`progress.csv`** under **`pg_eval_adapter_staged`**. Set **`AUTO_BASELINE=0`** before cell 1 if you already have **`baseline_staged`** and want to skip training there.


In [ ]:
# Code cell 1/7 — setup (repos, patch, run_one.sh, progress.csv)
# Drive must run in the notebook kernel (not from bash — Colab drive API needs the IPython kernel).
# Python fails on Colab ('NoneType' object has no attribute 'kernel').
# RUNBOOK_SETUP_REV=2026-03-28n  # fake_quant: current_clip clamp + b_max/s nan_to_num
import os
import subprocess

from pathlib import Path as _Path

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    _mp = _Path('/content/drive/MyDrive')
    if _mp.is_dir():
        print('Drive: already mounted at /content/drive (skipping drive.mount).')
    else:
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print('Drive mount skipped or failed:', e)

bash_script = r'''
# Code cell 1/7 — setup (repos, patch, run_one.sh, progress.csv)
set -eo pipefail
{ [ "${RUNBOOK_DEBUG:-0}" = "1" ] && set -x; } || true
FRESH_PROGRESS_CSV="${FRESH_PROGRESS_CSV:-0}"

BACKUP_ADAPTER='/content/drive/MyDrive/pg_ttadapter_staged'
BACKUP_BASELINE='/content/drive/MyDrive/baseline_staged'
if [ -d '/content/drive/MyDrive' ]; then
  mkdir -p "$BACKUP_ADAPTER" "$BACKUP_BASELINE" 2>/dev/null || echo 'RUNBOOK_WARN: could not mkdir Drive backup dirs (quota/mount); continuing — outputs stay under /content'
fi

[ -f /content/openai-pg/data/cached_challenge_fineweb.py ] || { rm -rf /content/openai-pg; git clone https://github.com/openai/parameter-golf.git /content/openai-pg; }
[ -f /content/pg_candidate/train_gpt.py ] || { rm -rf /content/pg_candidate; git clone https://github.com/jmoncayo-pursuit/parameter-golf-qat-int4.git /content/pg_candidate; }
[ -f /content/pg_ttadapter/train_gpt.py ] || { rm -rf /content/pg_ttadapter; git clone https://github.com/jmoncayo-pursuit/parameter-golf-qat-int4.git /content/pg_ttadapter; }

if git -C /content/pg_candidate fetch origin qat-int4-int6-gps-mlp; then
  git -C /content/pg_candidate checkout qat-int4-int6-gps-mlp 2>/dev/null || true
  if ! git -C /content/pg_candidate reset --hard origin/qat-int4-int6-gps-mlp; then
    echo 'RUNBOOK_FAIL: git reset --hard pg_candidate failed (corrupt/partial clone?). Try: rm -rf /content/pg_candidate && re-run cell 1.'
    exit 1
  fi
else
  echo 'WARN: git fetch pg_candidate failed; keeping existing tree'
fi

if git -C /content/pg_ttadapter fetch origin bayesian-backoff-cache-tt-adapter; then
  git -C /content/pg_ttadapter checkout bayesian-backoff-cache-tt-adapter 2>/dev/null || true
  if ! git -C /content/pg_ttadapter reset --hard origin/bayesian-backoff-cache-tt-adapter; then
    echo 'RUNBOOK_FAIL: git reset --hard pg_ttadapter failed (corrupt/partial clone?). Try: rm -rf /content/pg_ttadapter && re-run cell 1.'
    exit 1
  fi
else
  echo 'WARN: git fetch pg_ttadapter failed; keeping existing tree'
fi

python3 -m pip install -U pip -q || true
pip install -q -r /content/pg_candidate/requirements.txt zstandard sentencepiece || pip install -q -r /content/pg_candidate/requirements.txt zstandard sentencepiece || true

python3 /content/openai-pg/data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1 || \
  { sleep 3; python3 /content/openai-pg/data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1 || true; }

mkdir -p /content/pg_eval_adapter_staged

python3 - <<'PY'
from pathlib import Path

for repo in [Path('/content/pg_candidate'), Path('/content/pg_ttadapter')]:
    p = repo / 'train_gpt.py'
    s = p.read_text()
    old = 'with torch.autocast(device_type=device.type, dtype=torch.bfloat16):'
    new = 'with torch.autocast(device_type=device.type, dtype=(torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16), enabled=(device.type == "cuda")):'
    if new not in s:
        if old not in s:
            print(f'RUNBOOK_WARN: autocast snippet not found in {p}')
        else:
            s = s.replace(old, new)
    if repo.name == 'pg_candidate':
        old_prog = 'if rank == 0 and (bi // batch_seqs) % 50 == 0:'
        new_prog = 'if rank == 0 and (bi // batch_seqs) % 10 == 0:'
        if new_prog not in s:
            if old_prog not in s:
                print(f'RUNBOOK_WARN: candidate progress snippet not found in {p}')
            else:
                s = s.replace(old_prog, new_prog)
    old_clip = '    current_clip = 127 - alpha * (127 - clip_range)'
    new_clip = '    current_clip = (127 - alpha * (127 - clip_range)).clamp_min(1e-12)'
    if new_clip not in s and old_clip in s:
        s = s.replace(old_clip, new_clip, 1)
    old_scale = (
        '        b_max = w_p.view(-1, block_size).abs().amax(dim=1)\n'
        '        s = (b_max / current_clip).clamp_min(1e-12).to(torch.float16).clamp_min(torch.finfo(torch.float16).tiny)\n'
    )
    new_scale = (
        '        b_max = w_p.view(-1, block_size).abs().amax(dim=1)\n'
        '        b_max = torch.nan_to_num(b_max, nan=0.0)\n'
        '        s = (b_max / current_clip).clamp_min(1e-12).to(torch.float16).clamp_min(torch.finfo(torch.float16).tiny)\n'
        '        s = torch.nan_to_num(s, nan=torch.finfo(torch.float16).tiny, posinf=torch.finfo(torch.float16).max, neginf=torch.finfo(torch.float16).tiny)\n'
    )
    if 'torch.nan_to_num(b_max' not in s and old_scale in s:
        s = s.replace(old_scale, new_scale, 1)
    p.write_text(s)
PY

if [ ! -e /content/pg ]; then
  ln -s /content/pg_candidate /content/pg
  echo 'RUNBOOK: /content/pg -> /content/pg_candidate (checkpoint-only baseline uses same tree as candidate)'
fi

mkdir -p /content/pg/logs /content/pg/results/baseline_staged
PROG=/content/pg/results/baseline_staged/progress.csv
if [ "$FRESH_PROGRESS_CSV" = "1" ]; then
  rm -f "$PROG"
  if [ -d "/content/drive/MyDrive" ]; then
    rm -f "$BACKUP_BASELINE/progress.csv"
  fi
  echo 'RUNBOOK: FRESH_PROGRESS_CSV=1 — cleared baseline_staged progress.csv (local + Drive if mounted)'
fi

python3 - <<'PY'
from pathlib import Path
p = Path('/content/pg/train_gpt.py')
s = p.read_text()
old_sdpa_plain = (
    "    enable_cudnn_sdp(False)\n"
    "    enable_flash_sdp(True)\n"
    "    enable_mem_efficient_sdp(False)\n"
    "    enable_math_sdp(False)\n"
    "\n"
    "    logfile = None\n"
)
new_sdpa = (
    "    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n"
    "        enable_cudnn_sdp(False)\n"
    "        enable_flash_sdp(True)\n"
    "        enable_mem_efficient_sdp(False)\n"
    "        enable_math_sdp(False)\n"
    "    else:\n"
    "        enable_cudnn_sdp(False)\n"
    "        enable_flash_sdp(False)\n"
    "        enable_mem_efficient_sdp(True)\n"
    "        enable_math_sdp(True)\n"
    "\n"
    "    logfile = None\n"
)
old_sdpa_guarded = (
    "    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n"
    "        enable_cudnn_sdp(False)\n"
    "        enable_flash_sdp(True)\n"
    "        enable_mem_efficient_sdp(False)\n"
    "        enable_math_sdp(False)\n"
    "\n"
    "    logfile = None\n"
)
if new_sdpa in s:
    print('SDPA patch OK (already applied)')
elif old_sdpa_plain in s:
    s = s.replace(old_sdpa_plain, new_sdpa, 1)
    p.write_text(s)
    print('SDPA patch OK')
elif old_sdpa_guarded in s:
    s = s.replace(old_sdpa_guarded, new_sdpa, 1)
    p.write_text(s)
    print('SDPA patch OK')
else:
    print('RUNBOOK_WARN: SDPA block not found; train_gpt.py unchanged')
PY

python3 - <<'PY'
from pathlib import Path
p = Path('/content/pg/train_gpt.py')
s = p.read_text()
anchor = '    args = Hyperparameters()\n'
insert = '    args = Hyperparameters()\n    checkpoint_only = os.environ.get("CHECKPOINT_ONLY", "0") == "1"\n'
if 'checkpoint_only = os.environ.get("CHECKPOINT_ONLY", "0") == "1"' not in s:
    if anchor not in s:
        print('RUNBOOK_WARN: checkpoint_only anchor missing; skipping checkpoint patch')
    else:
        s = s.replace(anchor, insert, 1)
old = '        should_validate = last_step or (args.val_loss_every > 0 and step % args.val_loss_every == 0)\n'
new = '        should_validate = ((not checkpoint_only) and last_step) or (args.val_loss_every > 0 and step % args.val_loss_every == 0)\n'
co_ok = 'checkpoint_only = os.environ.get("CHECKPOINT_ONLY", "0") == "1"' in s
if new not in s:
    if old not in s:
        print('RUNBOOK_WARN: should_validate line missing; skipping that edit')
    elif not co_ok:
        print('RUNBOOK_WARN: skipping should_validate edit (checkpoint_only line not present)')
    else:
        s = s.replace(old, new, 1)
gate = '    quant_obj, quant_stats = quantize_state_dict_int8(base_model.state_dict(), clip_val_map=clip_val_map)'
exit_snip = 'checkpoint_only: exiting after final_model.pt'
mixed_hook = (
    '        log0(f"Total submission size: {model_bytes + code_bytes} bytes")\n'
    '\n'
    '    # Magnitude pruning: zero out smallest weights to improve compression'
)
mixed_hook_new = (
    '        log0(f"Total submission size: {model_bytes + code_bytes} bytes")\n'
    '\n'
    '    if checkpoint_only:\n'
    '        log0("checkpoint_only: exiting after final_model.pt")\n'
    '        return\n'
    '\n'
    '    # Magnitude pruning: zero out smallest weights to improve compression'
)
if exit_snip not in s and co_ok:
    if gate in s:
        s = s.replace(
            gate,
            '    if checkpoint_only:\n'
            '        log0("checkpoint_only: exiting after final_model.pt")\n'
            '        return\n\n'
            + gate,
            1,
        )
        print('CHECKPOINT_ONLY hook: int8 quantize gate')
    elif mixed_hook in s:
        s = s.replace(mixed_hook, mixed_hook_new, 1)
        print('CHECKPOINT_ONLY hook: post final_model.pt (mixed QAT / qat-int4-int6-gps-mlp)')
    else:
        print('RUNBOOK_WARN: CHECKPOINT_ONLY early exit not installed (no int8 or mixed-QAT hook)')
elif exit_snip in s:
    print('CHECKPOINT_ONLY hook already present')
elif not co_ok:
    print('RUNBOOK_WARN: skipping CHECKPOINT_ONLY hook (checkpoint_only line not present)')
p.write_text(s)
print('checkpoint_only patch step finished')
PY

cat >/content/pg/results/baseline_staged/run_one.sh <<'EOF_RUNNER'
#!/usr/bin/env bash
set -u
set -o pipefail

SEED="$1"
ITER="$2"
TAG="s${SEED}_i${ITER}"
PG_DIR="/content/pg"
OUT_DIR="/content/pg/results/baseline_staged"
BACKUP_DIR="${BACKUP_DIR:-/content/drive/MyDrive/baseline_staged}"

backup() {
  if [ -d "/content/drive/MyDrive" ]; then
    mkdir -p "${BACKUP_DIR}"
    cp -f "$1" "${BACKUP_DIR}/" 2>/dev/null || true
  fi
}

echo "==================== ${TAG} TRAIN ===================="
cd "${PG_DIR}"
rm -f final_model.pt

PYTHONUNBUFFERED=1 TORCHDYNAMO_DISABLE=1 CHECKPOINT_ONLY=1 \
DATA_PATH=/content/openai-pg/data/datasets/fineweb10B_sp1024 \
TOKENIZER_PATH=/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model \
VOCAB_SIZE=1024 \
ITERATIONS=${ITER} WARMUP_STEPS=0 VAL_LOSS_EVERY=0 EVAL_STRIDE=0 EVAL_CACHE=0 TRAIN_LOG_EVERY=10 \
TRAIN_BATCH_TOKENS=16384 TRAIN_SEQ_LEN=512 VAL_BATCH_SIZE=65536 \
MAX_WALLCLOCK_SECONDS=7200 COMPRESSOR=lzma SEED=${SEED} \
python train_gpt.py > "${PG_DIR}/logs/train_${TAG}.txt" 2>&1
train_rc=$?

backup "${PG_DIR}/logs/train_${TAG}.txt"

if [ $train_rc -ne 0 ] || [ ! -f "${PG_DIR}/final_model.pt" ]; then
  echo "${TAG},FAIL_TRAIN,,,,,,," >> "${OUT_DIR}/progress.csv"
  backup "${OUT_DIR}/progress.csv"
  echo "FAIL ${TAG} train rc=${train_rc}"
  exit 0
fi

cp "${PG_DIR}/final_model.pt" "${OUT_DIR}/final_model_${TAG}.pt"
backup "${OUT_DIR}/final_model_${TAG}.pt"

metrics=$(python3 - "${PG_DIR}/logs/train_${TAG}.txt" "${OUT_DIR}/final_model_${TAG}.pt" <<'PYSUM'
import re, sys, pathlib
log_path = pathlib.Path(sys.argv[1])
ckpt_path = pathlib.Path(sys.argv[2])
lines = log_path.read_text(errors='ignore').splitlines()
pat = re.compile(r"step:(\d+)/(\d+)\s+train_loss:([0-9.]+).*?step_avg:([0-9.]+)ms")
loss = step_avg = ''
for ln in lines:
    m = pat.search(ln)
    if m:
        loss, step_avg = m.group(3), m.group(4)
ckpt_bytes = str(ckpt_path.stat().st_size if ckpt_path.exists() else 0)
print(f"{loss}|{step_avg}|{ckpt_bytes}")
PYSUM
)
IFS='|' read -r last_loss step_avg_ms ckpt_bytes <<< "$metrics"
last_step="${ITER}"
total_steps="${ITER}"

echo "${TAG},OK,${last_step},${total_steps},${last_loss},${step_avg_ms},${ckpt_bytes}," >> "${OUT_DIR}/progress.csv"
backup "${OUT_DIR}/progress.csv"
echo "OK ${TAG} step=${last_step}/${total_steps} loss=${last_loss} step_avg_ms=${step_avg_ms} ckpt_bytes=${ckpt_bytes}"
EOF_RUNNER

chmod +x /content/pg/results/baseline_staged/run_one.sh

if [ "$FRESH_PROGRESS_CSV" != "1" ] && [ -f "$BACKUP_BASELINE/progress.csv" ] && [ ! -f "$PROG" ]; then
  cp -f "$BACKUP_BASELINE/progress.csv" "$PROG"
fi
if [ ! -f "$PROG" ]; then
  echo 'tag,status,last_step,total_steps,last_loss,step_avg_ms,ckpt_bytes,notes' > "$PROG"
fi

(
python3 - <<'PY'
import csv
from pathlib import Path
p = Path('/content/pg/results/baseline_staged/progress.csv')
try:
    rows = list(csv.DictReader(p.open(encoding='utf-8')))
except Exception as e:
    print('dedupe: could not read progress.csv:', e)
    rows = []
if not rows:
    print('dedupe: empty progress.csv (header only), skip dedupe')
else:
    def rank(r):
        ok = 1 if r.get('status') == 'OK' else 0
        complete = 1 if (r.get('last_step') or '').strip() and (r.get('last_step') == r.get('total_steps')) else 0
        has_loss = 1 if (r.get('last_loss') or '').strip() else 0
        return (ok, complete, has_loss)

    best = {}
    for r in rows:
        t = r['tag']
        if t not in best or rank(r) > rank(best[t]):
            best[t] = r

    FIELDS = ['tag','status','last_step','total_steps','last_loss','step_avg_ms','ckpt_bytes','notes']

    def _norm_baseline_row(r):
        r = dict(r)
        spill = r.pop(None, None)
        if spill is not None:
            bits = spill if isinstance(spill, list) else [spill]
            extra = 'overflow:' + '|'.join(str(x) for x in bits)
            prev = (r.get('notes') or '').strip()
            r['notes'] = (prev + ' | ' if prev else '') + extra
        return {k: (r.get(k) if r.get(k) is not None else '') for k in FIELDS}

    out = [_norm_baseline_row(best[k]) for k in sorted(best.keys())]
    with p.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['tag','status','last_step','total_steps','last_loss','step_avg_ms','ckpt_bytes','notes'])
        w.writeheader()
        w.writerows(out)
    print('deduped progress.csv rows:', len(out))
PY
) || { echo 'RUNBOOK_WARN: baseline progress dedupe failed (continuing without dedupe)'; true; }

AUTO_BASELINE="${AUTO_BASELINE:-1}"
BASE_HOME="/content/pg/results/baseline_staged"
if [ "$AUTO_BASELINE" = "1" ]; then
  need_train=0
  if ! grep -q ',OK,' "$PROG" 2>/dev/null; then
    need_train=1
  elif ! python3 - <<'PYCHK'
import csv, sys
from pathlib import Path
p = Path("/content/pg/results/baseline_staged/progress.csv")
bd = p.parent
if not p.is_file():
    sys.exit(1)
try:
    rows = list(csv.DictReader(p.open(encoding="utf-8")))
except Exception:
    sys.exit(1)
ok = [r for r in rows if r.get("status") == "OK"]
if not ok:
    sys.exit(1)
if any((bd / f"final_model_{r['tag']}.pt").is_file() for r in ok):
    sys.exit(0)
sys.exit(1)
PYCHK
  then
    need_train=1
    echo 'tag,status,last_step,total_steps,last_loss,step_avg_ms,ckpt_bytes,notes' > "$PROG"
    rm -f "$BASE_HOME"/final_model_*.pt 2>/dev/null || true
  fi

  if [ "$need_train" = "1" ]; then
    ITAUTO="${AUTO_BASELINE_ITERS:-100}"
    if python3 -c "import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)"; then
      /content/pg/results/baseline_staged/run_one.sh 42 "$ITAUTO"
    else
      echo 'RUNBOOK_WARN: no CUDA — cannot auto baseline. Use GPU runtime, mount Drive for MyDrive/baseline_staged, or AUTO_BASELINE=0 with local baseline_staged.'
    fi
  fi
fi

# When AUTO_BASELINE=1, fail the cell if Stage A cannot run (OK row + checkpoint .pt).
if [ "${AUTO_BASELINE:-1}" = "1" ]; then
  if ! python3 - <<'PYVERIFY'
import csv
from pathlib import Path
p = Path("/content/pg/results/baseline_staged/progress.csv")
bd = p.parent
if not p.is_file():
    raise SystemExit(1)
rows = list(csv.DictReader(p.open(encoding="utf-8")))
ok = [r for r in rows if r.get("status") == "OK"]
if not ok:
    raise SystemExit(1)
if not any((bd / "final_model_{}.pt".format(r["tag"])).is_file() for r in ok):
    raise SystemExit(1)
raise SystemExit(0)
PYVERIFY
  then
    if python3 -c "import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)"; then
      echo 'RUNBOOK_FAIL: baseline_staged has no OK row with final_model_<tag>.pt under /content/pg/results/baseline_staged. See /content/pg/logs/train_*.txt'
      exit 1
    else
      echo 'RUNBOOK_FAIL: GPU runtime required for AUTO_BASELINE=1 (no CUDA). Runtime → Change runtime type → GPU, or export AUTO_BASELINE=0 and restore MyDrive/baseline_staged.'
      exit 1
    fi
  fi
fi

cp -f "$PROG" "$BACKUP_BASELINE/progress.csv" 2>/dev/null || true

PROG_ADAPT='/content/pg_eval_adapter_staged/progress.csv'
if [ "$FRESH_PROGRESS_CSV" = '1' ]; then
  rm -f "$PROG_ADAPT"
  if [ -d '/content/drive/MyDrive' ]; then rm -f "$BACKUP_ADAPTER/progress.csv"; fi
  echo 'RUNBOOK: FRESH_PROGRESS_CSV=1 cleared adapter progress.csv'
fi
if [ "$FRESH_PROGRESS_CSV" != '1' ] && [ -f "$BACKUP_ADAPTER/progress.csv" ] && [ ! -f "$PROG_ADAPT" ]; then
  cp -f "$BACKUP_ADAPTER/progress.csv" "$PROG_ADAPT"
fi
if [ ! -f "$PROG_ADAPT" ]; then
  echo 'tag,status,offset,num_seqs,checkpoint_tag,candidate_bpb,ttadapter_bpb,delta_bpb,candidate_seconds,ttadapter_seconds,delta_seconds,notes' > "$PROG_ADAPT"
fi
cp -f "$PROG_ADAPT" "$BACKUP_ADAPTER/progress.csv" 2>/dev/null || true
echo "CELL1_OK backup=$BACKUP_ADAPTER"
cat "$PROG_ADAPT"

'''

_sub_env = os.environ.copy()
_sub_env.setdefault('GIT_TERMINAL_PROMPT', '0')
_sub_env.setdefault('PYTHONUNBUFFERED', '1')
_p = subprocess.Popen(
    ['bash', '-lc', bash_script],
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=_sub_env,
    bufsize=1,
    text=True,
)
_log_chunks = []
assert _p.stdout is not None
for _ln in _p.stdout:
    _log_chunks.append(_ln)
    print(_ln, end='', flush=True)
_rc = _p.wait()
_log = ''.join(_log_chunks)
if _rc != 0:
    _tail = _log[-12000:] if len(_log) > 12000 else _log
    raise RuntimeError(
        'bash setup exited with code %s. Output tail (look for Error, Traceback, RUNBOOK_FAIL, fatal):\n%s'
        % (_rc, _tail)
    )


---
## Code cell 2: Stage A — first 3 fixed validation slices
**Goal:** Define the reusable A/B helper once, then run it on three fixed token slices.

**Slices:** offsets `0`, `131072`, `262144`.

**Estimated time:** about 10–40 min.


In [ ]:
# CELL 2/7 - STAGE_A
import csv
import importlib.util
import os
import shutil
import time
from pathlib import Path

import sentencepiece as spm
import torch

BASELINE_CANDIDATES = [
    Path('/content/pg/results/baseline_staged'),
    Path('/content/drive/MyDrive/baseline_staged'),
    Path('/content/drive/MyDrive/pg_baseline_staged'),
]


def resolve_baseline_dir():
    for d in BASELINE_CANDIDATES:
        if (d / 'progress.csv').is_file():
            return d
    return None


STAGED_BACKUP = Path('/content/drive/MyDrive/pg_ttadapter_staged')
PROGRESS = Path('/content/pg_eval_adapter_staged/progress.csv')
DATA_PATH = Path('/content/openai-pg/data/datasets/fineweb10B_sp1024')
TOKENIZER_PATH = Path('/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model')
VOCAB_SIZE = 1024
TRAIN_SEQ_LEN = 512
EVAL_STRIDE = int(os.environ.get('EVAL_STRIDE', '64'))
EVAL_BATCH_SEQS = int(os.environ.get('EVAL_BATCH_SEQS', '8'))
CASES = [('off000000', 0, 32), ('off131072', 131072, 32), ('off262144', 262144, 32)]


def rows_by_tag():
    if not PROGRESS.exists():
        return {}
    try:
        with PROGRESS.open(encoding='utf-8', newline='') as f:
            return {r['tag']: r for r in csv.DictReader(f)}
    except Exception as e:
        print('⚠️ progress read failed:', e)
        return {}


def dedupe_progress():
    rows = []
    if PROGRESS.exists():
        try:
            with PROGRESS.open(encoding='utf-8', newline='') as f:
                rows = list(csv.DictReader(f))
        except Exception as e:
            print('⚠️ dedupe read failed:', e)
            return
    fields = [
        'tag', 'status', 'offset', 'num_seqs', 'checkpoint_tag',
        'candidate_bpb', 'ttadapter_bpb', 'delta_bpb',
        'candidate_seconds', 'ttadapter_seconds', 'delta_seconds', 'notes',
    ]

    def _norm_adapter_row(r):
        r = dict(r)
        spill = r.pop(None, None)
        if spill is not None:
            bits = spill if isinstance(spill, list) else [spill]
            extra = 'overflow:' + '|'.join(str(x) for x in bits)
            prev = (r.get('notes') or '').strip()
            r['notes'] = (prev + ' | ' if prev else '') + extra
        return {k: (r.get(k) if r.get(k) is not None else '') for k in fields}

    latest = {}
    for r in rows:
        latest[r['tag']] = r
    with PROGRESS.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(
            [
                _norm_adapter_row(latest[k])
                for k in sorted(
                    latest.keys(),
                    key=lambda t: int(latest[t]['offset']) if latest[t]['offset'] else 10**18,
                )
            ]
        )
    if STAGED_BACKUP.parent.exists():
        STAGED_BACKUP.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PROGRESS, STAGED_BACKUP / 'progress.csv')


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def select_checkpoint():
    bd = resolve_baseline_dir()
    if bd is None:
        tried = ', '.join(str(d / 'progress.csv') for d in BASELINE_CANDIDATES)
        raise FileNotFoundError(
            f'missing baseline progress.csv (tried: {tried}) — re-run cell 1 on GPU (auto-train), or restore MyDrive/baseline_staged from Drive'
        )
    bl = bd / 'progress.csv'
    with bl.open(encoding='utf-8', newline='') as f:
        rows = [r for r in csv.DictReader(f) if r.get('status') == 'OK']
    assert rows, 'no OK baseline checkpoint rows'
    with_ckpt = [r for r in rows if (bd / f'final_model_{r["tag"]}.pt').is_file()]
    if not with_ckpt:
        tags = [r['tag'] for r in rows]
        raise FileNotFoundError(
            f'baseline progress.csv has OK rows {tags} but no matching final_model_<tag>.pt under {bd}. '
            'Copy .pt files from Drive / prior runtime or re-run cell 1 (auto-baseline).'
        )
    best = max(with_ckpt, key=lambda r: (int(r['tag'].split('_i')[1]), -float(r['last_loss'])))
    ckpt = bd / f'final_model_{best["tag"]}.pt'
    return best['tag'], ckpt


def baseline_eval_ready():
    """True if baseline progress has at least one OK row with a local .pt (Run All must not crash)."""
    bd = resolve_baseline_dir()
    if bd is None:
        return False, 'no baseline progress.csv under known locations'
    bl = bd / 'progress.csv'
    try:
        with bl.open(encoding='utf-8', newline='') as f:
            rows = [r for r in csv.DictReader(f) if r.get('status') == 'OK']
    except Exception as e:
        return False, f'unreadable baseline progress.csv ({e})'
    with_ckpt = [r for r in rows if (bd / f'final_model_{r["tag"]}.pt').is_file()]
    if not with_ckpt:
        if rows:
            return (
                False,
                'OK rows in CSV but no matching final_model_<tag>.pt — re-run cell 1 on GPU',
            )
        return False, 'no OK rows in baseline progress.csv — re-run cell 1 on GPU'
    return True, ''


def build_model(mod, ckpt_path, device):
    args = mod.Hyperparameters()
    args.vocab_size = VOCAB_SIZE
    args.train_seq_len = TRAIN_SEQ_LEN
    model = mod.GPT(
        vocab_size=args.vocab_size,
        num_layers=args.num_layers,
        model_dim=args.model_dim,
        num_heads=args.num_heads,
        num_kv_heads=args.num_kv_heads,
        mlp_mult=args.mlp_mult,
        tie_embeddings=args.tie_embeddings,
        tied_embed_init_std=args.tied_embed_init_std,
        logit_softcap=args.logit_softcap,
        rope_base=args.rope_base,
        qk_gain_init=args.qk_gain_init,
        bigram_vocab_size=args.bigram_vocab_size,
        bigram_dim=args.bigram_dim,
    ).to(device)
    sd = torch.load(str(ckpt_path), map_location='cpu')
    model.load_state_dict(sd, strict=True)
    model.eval()
    return args, model


def run_case(tag, offset, num_seqs):
    existing = rows_by_tag().get(tag)
    if existing and existing.get('status') == 'OK':
        print(f'SKIP {tag} already OK')
        return

    ckpt_tag, ckpt = select_checkpoint()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    candidate = load_module('candidate_tg', '/content/pg_candidate/train_gpt.py')
    ttadapter = load_module('ttadapter_tg', '/content/pg_ttadapter/train_gpt.py')
    sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_PATH))
    assert int(sp.vocab_size()) == VOCAB_SIZE

    val_full = candidate.load_validation_tokens(str(DATA_PATH / 'fineweb_val_*.bin'), TRAIN_SEQ_LEN)
    need = num_seqs * TRAIN_SEQ_LEN + 1
    val_tokens = val_full[offset:offset + need].to(dtype=torch.int64)
    assert val_tokens.numel() == need, (offset, need, val_tokens.numel())
    luts = candidate.build_sentencepiece_luts(sp, VOCAB_SIZE, device)

    print(f'\n=== RUN {tag} offset={offset} num_seqs={num_seqs} checkpoint={ckpt_tag} ===')
    cand_args, cand_model = build_model(candidate, ckpt, device)
    t0 = time.perf_counter()
    _, cand_bpb = candidate.eval_val_sliding_cached(
        cand_args, cand_model, 0, 1, device, val_tokens, *luts,
        stride=EVAL_STRIDE, batch_seqs=EVAL_BATCH_SEQS,
    )
    cand_seconds = time.perf_counter() - t0
    print(f'candidate_bpb:{cand_bpb:.6f} candidate_seconds:{cand_seconds:.2f}')
    del cand_model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    tt_args, tt_model = build_model(ttadapter, ckpt, device)
    t1 = time.perf_counter()
    _, tt_bpb = ttadapter.eval_val_sliding_cached(
        tt_args, tt_model, 0, 1, device, val_tokens, *luts,
        stride=EVAL_STRIDE, batch_seqs=EVAL_BATCH_SEQS,
    )
    tt_seconds = time.perf_counter() - t1
    print(f'ttadapter_bpb:{tt_bpb:.6f} ttadapter_seconds:{tt_seconds:.2f}')

    delta_bpb = tt_bpb - cand_bpb
    delta_seconds = tt_seconds - cand_seconds
    print(f'delta_bpb:{delta_bpb:+.6f} delta_seconds:{delta_seconds:+.2f}')

    row = {
        'tag': tag,
        'status': 'OK',
        'offset': str(offset),
        'num_seqs': str(num_seqs),
        'checkpoint_tag': ckpt_tag,
        'candidate_bpb': f'{cand_bpb:.6f}',
        'ttadapter_bpb': f'{tt_bpb:.6f}',
        'delta_bpb': f'{delta_bpb:+.6f}',
        'candidate_seconds': f'{cand_seconds:.2f}',
        'ttadapter_seconds': f'{tt_seconds:.2f}',
        'delta_seconds': f'{delta_seconds:+.2f}',
        'notes': '',
    }
    rows = []
    if PROGRESS.exists():
        with PROGRESS.open(encoding='utf-8', newline='') as f:
            rows = list(csv.DictReader(f))
    rows = [r for r in rows if r['tag'] != tag]
    rows.append(row)
    with PROGRESS.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        w.writeheader()
        w.writerows(rows)
    dedupe_progress()
    print('--- progress so far ---')
    print(PROGRESS.read_text(encoding='utf-8', errors='replace'))


TG_C = Path('/content/pg_candidate/train_gpt.py')
TG_T = Path('/content/pg_ttadapter/train_gpt.py')
BASELINE_DIR = resolve_baseline_dir()

if not TG_C.is_file() or not TG_T.is_file():
    print('⏭️ SKIP Stage A: run cell 1 first (train_gpt.py missing).')
elif not torch.cuda.is_available():
    print('⏭️ SKIP Stage A: no CUDA (Runtime → GPU).')
elif BASELINE_DIR is None:
    tried = ', '.join(str(d / 'progress.csv') for d in BASELINE_CANDIDATES)
    print(
        '⏭️ SKIP Stage A: no baseline progress.csv (tried: ' + tried + '). Re-run cell 1 on GPU, or restore baseline_staged to Drive/local.'
    )
else:
    ok_go, reason = baseline_eval_ready()
    if not ok_go:
        print('⏭️ SKIP Stage A:', reason)
    else:
        for tag, offset, num_seqs in CASES:
            run_case(tag, offset, num_seqs)


---
## Code cell 3: Stage A decision
**Goal:** Decide whether to continue to extra slices by checking that the first 3 runs finished cleanly, the adapter was not catastrophically worse, and at least 2 of the 3 slices show non-positive `delta_bpb`.

**Estimated time:** under 1 min.


In [ ]:
# CELL 3/7 - STAGE_A_DECISION
import csv
import shutil
from pathlib import Path

PROGRESS = Path('/content/pg_eval_adapter_staged/progress.csv')
FLAG = Path('/content/pg_eval_adapter_staged/proceed_stage_b.txt')
stage_a_tags = {'off000000', 'off131072', 'off262144'}
if not PROGRESS.exists():
    rows = []
    print('⚠️ no progress.csv yet (run cell 1).')
else:
    try:
        with PROGRESS.open(encoding='utf-8', newline='') as f:
            rows = list(csv.DictReader(f))
    except Exception as e:
        print('⚠️ could not read progress.csv:', e)
        rows = []
stage_a = [r for r in rows if r.get('tag') in stage_a_tags]


def _delta(r):
    try:
        return float(r.get('delta_bpb', 'nan'))
    except (TypeError, ValueError):
        return float('nan')


ok_rows = [r for r in stage_a if r.get('status') == 'OK']
non_positive = [r for r in ok_rows if _delta(r) <= 0.0]
catastrophic = [r for r in ok_rows if _delta(r) > 0.010]
proceed_stage_b = len(ok_rows) == 3 and len(non_positive) >= 2 and len(catastrophic) == 0

print('Stage A rows:')
for r in stage_a:
    print(r)
print('\nnon_positive:', len(non_positive), 'catastrophic:', len(catastrophic))
print('PROCEED_STAGE_B:', 'YES' if proceed_stage_b else 'NO')
FLAG.write_text('YES\n' if proceed_stage_b else 'NO\n')

bd = Path('/content/drive/MyDrive/pg_ttadapter_staged')
if bd.parent.exists():
    try:
        bd.mkdir(parents=True, exist_ok=True)
        if PROGRESS.exists():
            shutil.copy(PROGRESS, bd / 'progress.csv')
        (bd / 'proceed_stage_b.txt').write_text('YES\n' if proceed_stage_b else 'NO\n')
    except OSError as e:
        print('⚠️ Drive backup skipped:', e)


---
## Code cell 4: Stage B — 4 more fixed validation slices
**Goal:** Repeat the same A/B on four additional token slices only if Stage A says the direction is still worth checking.

**Slices:** offsets `393216`, `524288`, `655360`, `786432`.

**Estimated time:** about 15–50 min if it runs.


In [ ]:
# CELL 4/7 - STAGE_B
from pathlib import Path

import torch

FLAG = Path('/content/pg_eval_adapter_staged/proceed_stage_b.txt')
CASES = [('off393216', 393216, 32), ('off524288', 524288, 32), ('off655360', 655360, 32), ('off786432', 786432, 32)]
if not FLAG.exists() or FLAG.read_text().strip() != 'YES':
    print('Stage B skipped. Run Cell 3 first and continue only if PROCEED_STAGE_B: YES.')
elif not torch.cuda.is_available():
    print('⏭️ SKIP Stage B: no CUDA.')
elif not callable(globals().get('run_case')):
    print('⏭️ SKIP Stage B: run_case missing — re-run Stage A cell (cell 2).')
else:
    for tag, offset, num_seqs in CASES:
        run_case(tag, offset, num_seqs)


---
## Code cell 5: TT_ADAPTER_SIGNAL_STRONG
**Goal:** Print the aggregate mini-`val_bpb` signal from the staged slices and keep the gate honest.

**Interpretation:** this is a Colab mini-slice signal for the branch question, not a final H100 ruling.

**Estimated time:** under 1 min.


In [ ]:
# CELL 5/7 - TT_ADAPTER_SIGNAL_STRONG
import csv
import statistics
from pathlib import Path

PROGRESS = Path('/content/pg_eval_adapter_staged/progress.csv')
if not PROGRESS.exists():
    print('⚠️ no progress.csv (run cell 1).')
    rows = []
else:
    try:
        with PROGRESS.open(encoding='utf-8', newline='') as f:
            rows = list(csv.DictReader(f))
    except Exception as e:
        print('⚠️ could not read progress.csv:', e)
        rows = []
ok = [r for r in rows if r.get('status') == 'OK']

if not ok:
    print('TT_ADAPTER_SIGNAL_STRONG: NO (no successful rows)')
else:
    deltas = [float(r['delta_bpb']) for r in ok]
    gains = [d for d in deltas if d < 0.0]
    catastrophic = [d for d in deltas if d > 0.010]
    candidate_seconds = [float(r['candidate_seconds']) for r in ok]
    tt_seconds = [float(r['ttadapter_seconds']) for r in ok]
    strong_signal = len(catastrophic) == 0 and len(gains) >= max(3, (len(ok) + 1) // 2) and statistics.median(deltas) < 0.0

    print(f'successful_rows={len(ok)} gains={len(gains)} catastrophic={len(catastrophic)}')
    print(f'delta_bpb_best={min(deltas):+.6f} delta_bpb_median={statistics.median(deltas):+.6f} delta_bpb_worst={max(deltas):+.6f}')
    print(f'candidate_seconds_median={statistics.median(candidate_seconds):.2f} ttadapter_seconds_median={statistics.median(tt_seconds):.2f}')
    print('TT_ADAPTER_SIGNAL_STRONG:', 'YES' if strong_signal else 'NO')

print('\nFull table:')
for r in rows:
    print(r)


---
## Code cell 6: Table
**Goal:** Pretty-print `progress.csv` so each slice-level A/B is easy to scan.

**Estimated time:** seconds.


In [ ]:
%%bash
PROG=/content/pg_eval_adapter_staged/progress.csv
if [ -f "$PROG" ]; then column -s, -t < "$PROG"; else echo '(no progress.csv — run cell 1)'; fi


---
## Code cell 7: Final readout
**Goal:** Print the strongest available slice-level result, weakest result, and the shared checkpoint tag.

**Estimated time:** under 1 min.


In [ ]:
# CELL 7/7 - FINAL_READOUT
import csv
from pathlib import Path

PROGRESS = Path('/content/pg_eval_adapter_staged/progress.csv')
rows = []
if not PROGRESS.exists():
    print('⏭️ SKIP final readout: no progress.csv (run cell 1).')
else:
    try:
        with PROGRESS.open(encoding='utf-8', newline='') as f:
            rows = [r for r in csv.DictReader(f) if r.get('status') == 'OK']
    except Exception as e:
        print('⚠️ could not read progress.csv:', e)
        rows = []
    if not rows:
        print('⏭️ SKIP final readout: no successful rows.')
    else:
        best = min(rows, key=lambda r: float(r['delta_bpb']))
        worst = max(rows, key=lambda r: float(r['delta_bpb']))
        ckpt_tags = sorted({r['checkpoint_tag'] for r in rows if r.get('checkpoint_tag')})

        print('=== FINAL READOUT ===')
        print('checkpoint_tags:', ', '.join(ckpt_tags) if ckpt_tags else '(missing)')
        print('best_slice:', best['tag'], 'offset=', best['offset'], 'delta_bpb=', best['delta_bpb'], 'candidate_bpb=', best['candidate_bpb'], 'ttadapter_bpb=', best['ttadapter_bpb'])
        print('worst_slice:', worst['tag'], 'offset=', worst['offset'], 'delta_bpb=', worst['delta_bpb'], 'candidate_bpb=', worst['candidate_bpb'], 'ttadapter_bpb=', worst['ttadapter_bpb'])
        print('successful_rows:', len(rows))
